# Daily Standardized Streamflow Index (SSI)

This notebook computes the **Standardized Streamflow Index (SSI)** at daily resolution for 33 gauging stations,
following the flexible distribution-fitting methodology of Vicente-Serrano et al. (*FlexDroughtIndex*).

### Key methodological decisions
- **SSI = SPI applied to streamflow**: streamflow is bottom-truncated at 0, so we use the same family of positive-valued distributions as SPI.
- **No time-scale aggregation**: we work directly on daily Q — no rolling means on the input series.
- **Day-of-year (DOY) ±15 day pooling window**: because each DOY has only ~60 observations (one per year), we pool values from DOY−15 to DOY+15 across all years (~1,860 obs) to get statistically robust distribution fits.
- **6 candidate distributions**: Gamma, Log-Normal, Weibull, Pearson III, Log-Logistic, GEV.
- **Selection criterion**: Shapiro–Wilks W statistic on the normalized output — the distribution that produces the most normal SSI series wins.
- **Zero-flow adjustment**: if a fraction *p₀* of pooled values equals 0, the CDF is shifted to account for the point mass at zero.

### Output
A long-format DataFrame with columns: `station_id`, `date`, `Q`, `SSI`.

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
# Standard scientific stack + scipy.stats for distribution fitting
# and Shapiro-Wilks test. tqdm provides per-station progress bars.

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import shapiro
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')   # suppress scipy convergence warnings

print('Libraries loaded successfully.')

In [ ]:
# ── Cell 2: Load and inspect the data ────────────────────────────────────────
# The CSV contains daily imputed streamflow (Q_imp) for 33 stations
# covering 1961-01-01 to 2020-12-31.
# We parse 'date' as datetime upfront and sort by station + date.

DATA_PATH = 'data/caudales_diarios_imputados_CORE_FILTRADO.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values(['station_id', 'date']).reset_index(drop=True)

print(f'Shape       : {df.shape}')
print(f'Stations    : {df["station_id"].nunique()}')
print(f'Date range  : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'\nQ_imp stats:')
print(df['Q_imp'].describe().round(2))
print(f'\nZero flows  : {(df["Q_imp"] == 0).sum()} ({(df["Q_imp"] == 0).mean()*100:.2f}%)')
print(f'NaN flows   : {df["Q_imp"].isna().sum()}')
df.head()

## Methodology: distribution fitting functions

For each station × DOY we:
1. Collect the **pooled sample** — all historical Q values with |DOY_i − DOY_target| ≤ 15 (circular, wrapping around year boundaries).
2. Fit each of the **6 distributions** via MLE (scipy `dist.fit`).
3. Transform the pooled sample to the standard normal via the fitted CDF → `norm.ppf`.
4. Run **Shapiro–Wilks** on the transformed values; record the W statistic.
5. Pick the distribution with the **highest W** (closest to normality).
6. Apply that distribution to the actual DOY-specific observations to produce the final SSI values.

In [ ]:
# ── Cell 3: Define the six candidate distributions ────────────────────────────
# These are the same distribution families commonly used in SPI/SSI literature
# and consistent with the FlexDroughtIndex R package for bottom-truncated data.
#
#   Gamma       — classic SPI distribution; shape + scale
#   Log-Normal  — heavy right tail; common for streamflow
#   Weibull     — flexible; good for low-flow regimes
#   Pearson III — 3-parameter; used in SPEI (unbounded) but valid here too
#   Log-Logistic (Fisk) — often outperforms Gamma for skewed hydro data
#   GEV         — extreme-value family; captures heavy tails

DISTRIBUTIONS = {
    'Gamma'      : stats.gamma,
    'LogNormal'  : stats.lognorm,
    'Weibull'    : stats.weibull_min,
    'PearsonIII' : stats.pearson3,
    'LogLogistic': stats.fisk,
    'GEV'        : stats.genextreme,
}

print('Candidate distributions:')
for name, dist in DISTRIBUTIONS.items():
    print(f'  {name:<12} → scipy.stats.{dist.name}')

In [ ]:
# ── Cell 4: Helper functions ──────────────────────────────────────────────────

def doy_circular_distance(doy_array, target_doy):
    """
    Circular distance between an array of DOY values and a target DOY.
    DOY wraps at 365 (e.g., distance between DOY 1 and DOY 360 = 6).
    This handles pooling windows that straddle the year boundary.
    """
    diff = np.abs(doy_array.astype(int) - int(target_doy))
    return np.minimum(diff, 365 - diff)


def fit_and_score(dist, values):
    """
    Fit a scipy distribution to `values` and return (params, W_statistic).

    Steps:
      1. MLE fit via dist.fit(values)
      2. CDF → probability integral transform → standard normal
      3. Shapiro–Wilks W on the transformed series

    Returns (-inf) W if the fit fails or produces degenerate results.
    """
    try:
        params = dist.fit(values)
        p = dist.cdf(values, *params)
        # Clip to avoid ±inf after norm.ppf
        p = np.clip(p, 1e-6, 1 - 1e-6)
        z = stats.norm.ppf(p)
        # Shapiro–Wilks requires 3 ≤ n ≤ 5000; pool size is ~1860 → fine
        w, _ = shapiro(z)
        return params, float(w)
    except Exception:
        return None, -np.inf


print('Helper functions defined.')

In [ ]:
# ── Cell 5: Core SSI computation function (single station) ────────────────────
#
# For each unique DOY in the series:
#   a) Pool all Q values within circular DOY ± WINDOW days across all years.
#   b) Handle zero-flow: if proportion p0 > 0, fit only on positive values
#      and shift the CDF: F_adj(q) = p0 + (1-p0)*F(q)  for q > 0
#                         F_adj(0) = p0 * uniform ≈ p0/2  (midpoint)
#   c) Fit 6 distributions on the pool; select best by Shapiro-Wilks W.
#   d) Apply the best-fit CDF to the actual DOY observations → norm.ppf → SSI.
#
# Returns:
#   ssi        : pd.Series aligned to input index
#   stats_log  : dict {doy: {'best': str, 'W': {dist_name: float}}}

WINDOW = 15   # ±15 days pooling window

def compute_ssi_station(series, window=WINDOW):
    """
    Compute daily SSI for a single station.

    Parameters
    ----------
    series : pd.Series
        Daily Q values with a DatetimeIndex. May contain NaNs.
    window : int
        Half-width of the DOY pooling window in days (default 15).

    Returns
    -------
    ssi : pd.Series  (same index as input, float)
    stats_log : dict
    """
    doy_arr  = series.index.dayofyear          # integer 1-366
    ssi      = pd.Series(np.nan, index=series.index, name='SSI')
    stats_log = {}

    for d in sorted(series.index.dayofyear.unique()):

        # ── (a) Build pool ────────────────────────────────────────────────
        mask_pool  = doy_circular_distance(doy_arr, d) <= window
        pool_vals  = series[mask_pool].dropna().values

        if len(pool_vals) < 30:          # safeguard: skip if too few obs
            continue

        # ── (b) Zero-flow probability mass ───────────────────────────────
        p0        = float(np.mean(pool_vals == 0))
        fit_vals  = pool_vals[pool_vals > 0] if p0 > 0 else pool_vals

        if len(fit_vals) < 10:
            continue

        # ── (c) Fit all distributions; select best by Shapiro-Wilks W ───
        best_name, best_params, best_w = None, None, -np.inf
        sw_scores = {}

        for name, dist in DISTRIBUTIONS.items():
            params, w = fit_and_score(dist, fit_vals)
            sw_scores[name] = round(w, 5)
            if w > best_w:
                best_w, best_name, best_params = w, name, params

        stats_log[d] = {'best': best_name, 'W': sw_scores}

        if best_params is None:
            continue

        # ── (d) Apply best CDF to actual DOY observations ────────────────
        mask_doy    = (doy_arr == d)
        actual      = series[mask_doy].dropna()

        if len(actual) == 0:
            continue

        best_dist = DISTRIBUTIONS[best_name]
        p = best_dist.cdf(actual.values, *best_params)

        # Zero-flow adjustment on the output probabilities
        if p0 > 0:
            p = np.where(
                actual.values == 0,
                p0 / 2,                          # midpoint of the zero mass
                p0 + (1.0 - p0) * p              # shift CDF for positive Q
            )

        p = np.clip(p, 1e-6, 1 - 1e-6)
        ssi[actual.index] = stats.norm.ppf(p)

    return ssi, stats_log


print(f'SSI function defined. Pooling window: ±{WINDOW} days.')

In [ ]:
# ── Cell 6: Run SSI computation for all 33 stations ──────────────────────────
#
# For each station we:
#   1. Extract and index its time series.
#   2. Call compute_ssi_station() → SSI series + distribution stats.
#   3. Append results to a list (avoids repeated DataFrame concatenation).
#
# Expected runtime: a few minutes depending on machine speed.

records    = []       # will hold one dict per station
all_stats  = {}       # {station_id: stats_log}

stations = df['station_id'].unique()

for station in tqdm(stations, desc='Computing SSI'):

    sub = (
        df[df['station_id'] == station]
        .set_index('date')['Q_imp']
        .sort_index()
    )

    ssi, stats_log = compute_ssi_station(sub, window=WINDOW)

    records.append(pd.DataFrame({
        'station_id': station,
        'date'      : sub.index,
        'Q'         : sub.values,
        'SSI'       : ssi.values,
    }))

    all_stats[station] = stats_log

print(f'\nDone. Processed {len(stations)} stations.')

In [ ]:
# ── Cell 7: Assemble final DataFrame and sanity checks ────────────────────────
#
# Concatenate all per-station DataFrames into the final long-format table.
# We also run basic checks:
#   - NaN rate in SSI (should be low / only where Q was NaN)
#   - SSI mean ≈ 0 and std ≈ 1 per station (normality check)
#   - Distribution frequency across all stations × DOYs

final_df = (
    pd.concat(records, ignore_index=True)
    [['station_id', 'date', 'Q', 'SSI']]   # enforce column order
)

print('=== Final DataFrame ===' )
print(f'Shape   : {final_df.shape}')
print(f'Columns : {list(final_df.columns)}')
print(f'NaN SSI : {final_df["SSI"].isna().sum()} ({final_df["SSI"].isna().mean()*100:.2f}%)')
print()

print('=== SSI descriptive stats per station (first 10) ===')
ssi_stats = (
    final_df.groupby('station_id')['SSI']
    .agg(['mean', 'std', 'min', 'max'])
    .round(3)
)
print(ssi_stats.head(10))
print()

print('=== Best distribution frequency across all stations × DOYs ===')
best_dist_counts = {d: 0 for d in DISTRIBUTIONS}
for station, slog in all_stats.items():
    for doy, info in slog.items():
        if info['best'] in best_dist_counts:
            best_dist_counts[info['best']] += 1
total_fits = sum(best_dist_counts.values())
for name, cnt in sorted(best_dist_counts.items(), key=lambda x: -x[1]):
    print(f'  {name:<12}: {cnt:>5}  ({cnt/total_fits*100:.1f}%)')

final_df.head(10)

In [ ]:
# ── Cell 8: Save outputs ──────────────────────────────────────────────────────
#
# Primary output: long-format CSV with station_id, date, Q, SSI.
# Secondary output: distribution selection log as a separate CSV
#   (station_id, doy, best_dist, W_Gamma, W_LogNormal, ...) for diagnostics.

# --- Main results ---
OUT_SSI = 'data/SSI_daily.csv'
final_df.to_csv(OUT_SSI, index=False)
print(f'SSI results saved → {OUT_SSI}')

# --- Distribution selection log ---
log_rows = []
for station, slog in all_stats.items():
    for doy, info in slog.items():
        row = {'station_id': station, 'doy': doy, 'best_dist': info['best']}
        row.update({f'W_{k}': v for k, v in info['W'].items()})
        log_rows.append(row)

dist_log_df = pd.DataFrame(log_rows)
OUT_LOG = 'data/SSI_distribution_log.csv'
dist_log_df.to_csv(OUT_LOG, index=False)
print(f'Distribution log saved → {OUT_LOG}')
print(f'\nDistribution log shape: {dist_log_df.shape}')
dist_log_df.head()